
ORDER BY
--------


    •	Used to sort the data
	•	Sorting happens globally (across all partitions)
	•	Expensive operation → causes shuffle

df.orderBy("column_name")                  # Ascending (default)
df.orderBy("column_name", ascending=False) # Descending

# Multiple columns
df.orderBy(["col1", "col2"], ascending=[True, False])

from pyspark.sql import functions as F

df.orderBy("salary").show()               # Low → High
df.orderBy(F.col("salary").desc()).show() # High → Low

Point       Explanation
Shuffle     Yes (expensive)
Scope       Global sorting
Use case    Final output, reporting
Alternative sortWithinPartitions() (faster, local sort)

df.orderBy(F.col("salary").desc()).limit(10)


GROUP BY 

    •	Used to group rows based on column(s)
	•	Always used with aggregation
	•	Creates GroupedData object

df.groupBy("column")

# Multiple columns
df.groupBy("col1", "col2")

df.groupBy("department").count().show()

Point       Explanation
Returns     GroupedData (not DataFrame yet)
Needs       Aggregation function
Shuffle     Yes
Use case    Summarization

Total sales per region:

df.groupBy("region").sum("sales")


AGGREGATE FUNCTIONS 

Used to perform calculations on grouped data:
	•	sum
	•	avg
	•	count
	•	max
	•	min

df.groupBy("col").agg(F.sum("col2"))

Multiple Aggregations

df.groupBy("dept").agg(
    F.sum("salary").alias("total_salary"),
    F.avg("salary").alias("avg_salary"),
    F.max("salary").alias("max_salary")
)

Without groupBy (Global Aggregation)

df.agg(F.sum("salary"), F.avg("salary"))

df.groupBy("department").agg(
    F.count("*").alias("emp_count"),
    F.sum("salary").alias("total_salary")
).show()



In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .appName("Joins")
    .master("local[*]")   
    .getOrCreate()
)

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/27 21:48:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
from pyspark.sql import functions as F

data = [
    (1, "Alice", "HR", "Mumbai", 50000, "2024-01-10"),
    (2, "Bob", "IT", "Bangalore", 70000, "2024-01-12"),
    (3, "Charlie", "IT", "Mumbai", 80000, "2024-01-15"),
    (4, "David", "HR", "Delhi", 60000, "2024-01-18"),
    (5, "Eve", "Sales", "Mumbai", 75000, "2024-01-20"),
    (6, "Frank", "Sales", "Delhi", 72000, "2024-01-22"),
    (7, "Grace", "IT", "Bangalore", 90000, "2024-01-25"),
    (8, "Helen", "HR", "Mumbai", 65000, "2024-01-28")
]

columns = ["id", "name", "department", "city", "salary", "joining_date"]

df = spark.createDataFrame(data, columns)

In [3]:
df.show(3)

+---+-------+----------+---------+------+------------+
| id|   name|department|     city|salary|joining_date|
+---+-------+----------+---------+------+------------+
|  1|  Alice|        HR|   Mumbai| 50000|  2024-01-10|
|  2|    Bob|        IT|Bangalore| 70000|  2024-01-12|
|  3|Charlie|        IT|   Mumbai| 80000|  2024-01-15|
+---+-------+----------+---------+------+------------+
only showing top 3 rows



LEVEL 1 — Basic 

1. Count total number of employees
2. Count employees per department
3. Find total salary per department
4. Find average salary per department
5. Find maximum salary per department
6. Find minimum salary per department
7. Sort employees by salary (ascending)
8. Sort employees by salary (descending)
9. Get top 3 highest paid employees
10. Count employees per city

LEVEL 2 — Intermediate 

11. Find departments where average salary > 70000

12. Find total salary per city

13. Sort departments based on total salary (descending)

14. Find number of employees in each department and sort by count

15. Find department with highest total salary

16. Find city with lowest average salary

17. Find top 2 departments by average salary

18. Perform multiple aggregations:
	•	total salary
	•	average salary
	•	employee count
19. Global aggregations:
	•	total salary
	•	average salary
	•	max salary
20. Sort employees first by department, then salary descending

Interview 

21. Find departments where employee count > 2

Requires:
	•	groupBy
	•	filter

22. Find employees earning more than department average

23. Find top earning employee in each department
24. Find second highest salary per department

25. Real-Time Scenario

“Daily business report”

Find:
	•	total salary per department per city
	•	sort by highest total salary
    


In [ ]:
from pyspark.sql.functions import col
# 1. Count total number of employees
# 2. Count employees per department

# Top 2 depmartments by total sal 

# 9. Get top 3 highest paid employees
# 10. Count employees per city

#df.count()
#df.groupby("department").count().show()
df.groupby("department").agg(
    F.sum("salary").alias("total_sal"))
    .orderby(F.col("total_sal").desc())
).limit(2).show()


TypeError: 'Column' object is not callable

In [12]:
df.groupBy("department").agg(
    F.sum("salary").alias("total_sal")
).orderBy(
    F.col("total_sal").desc()
).limit(2).show()

+----------+---------+
|department|total_sal|
+----------+---------+
|        IT|   240000|
|        HR|   175000|
+----------+---------+



In [14]:
#df.orderby("salary").asc().show()
df.orderBy(col("salary").asc()).show()

+---+-------+----------+---------+------+------------+
| id|   name|department|     city|salary|joining_date|
+---+-------+----------+---------+------+------------+
|  1|  Alice|        HR|   Mumbai| 50000|  2024-01-10|
|  4|  David|        HR|    Delhi| 60000|  2024-01-18|
|  8|  Helen|        HR|   Mumbai| 65000|  2024-01-28|
|  2|    Bob|        IT|Bangalore| 70000|  2024-01-12|
|  6|  Frank|     Sales|    Delhi| 72000|  2024-01-22|
|  5|    Eve|     Sales|   Mumbai| 75000|  2024-01-20|
|  3|Charlie|        IT|   Mumbai| 80000|  2024-01-15|
|  7|  Grace|        IT|Bangalore| 90000|  2024-01-25|
+---+-------+----------+---------+------+------------+



In [19]:
sort_col="salary"
df.orderBy(col(sort_col).desc()).show()

+---+-------+----------+---------+------+------------+
| id|   name|department|     city|salary|joining_date|
+---+-------+----------+---------+------+------------+
|  7|  Grace|        IT|Bangalore| 90000|  2024-01-25|
|  3|Charlie|        IT|   Mumbai| 80000|  2024-01-15|
|  5|    Eve|     Sales|   Mumbai| 75000|  2024-01-20|
|  6|  Frank|     Sales|    Delhi| 72000|  2024-01-22|
|  2|    Bob|        IT|Bangalore| 70000|  2024-01-12|
|  8|  Helen|        HR|   Mumbai| 65000|  2024-01-28|
|  4|  David|        HR|    Delhi| 60000|  2024-01-18|
|  1|  Alice|        HR|   Mumbai| 50000|  2024-01-10|
+---+-------+----------+---------+------+------------+



In [21]:
#11. Find departments where average salary > 70000
df.groupBy("department").agg(
    F.avg("salary").alias("avg_sal")
).filter(F.col("avg_sal")>70000).show()

+----------+-------+
|department|avg_sal|
+----------+-------+
|        IT|80000.0|
|     Sales|73500.0|
+----------+-------+



In [24]:
#16. Find city with lowest average salary
from pyspark.sql import functions as F 
min_sal_per_city=df.groupBy("city").agg(
    F.avg("salary").alias("avg_sal")
).orderBy(
    F.col("avg_sal").asc()
).limit(1)

In [25]:
min_sal_per_city.show()

+-----+-------+
| city|avg_sal|
+-----+-------+
|Delhi|66000.0|
+-----+-------+



In [ ]:
# 25. Real-Time Scenario

# “Daily business report”

# Find:
# 	•	total salary per department per city
# 	•	sort by highest total salary



In [26]:
df.show()

+---+-------+----------+---------+------+------------+
| id|   name|department|     city|salary|joining_date|
+---+-------+----------+---------+------+------------+
|  1|  Alice|        HR|   Mumbai| 50000|  2024-01-10|
|  2|    Bob|        IT|Bangalore| 70000|  2024-01-12|
|  3|Charlie|        IT|   Mumbai| 80000|  2024-01-15|
|  4|  David|        HR|    Delhi| 60000|  2024-01-18|
|  5|    Eve|     Sales|   Mumbai| 75000|  2024-01-20|
|  6|  Frank|     Sales|    Delhi| 72000|  2024-01-22|
|  7|  Grace|        IT|Bangalore| 90000|  2024-01-25|
|  8|  Helen|        HR|   Mumbai| 65000|  2024-01-28|
+---+-------+----------+---------+------+------------+



In [28]:
# Find total sal per dept 
# calculate % contribution of each dept to overall salary 

dept_df=df.groupBy("department").agg(
    F.sum("salary").alias("total_Sal")
)
total_df=df.agg(
    F.sum("salary").alias("overall_sal")
)
result_df=dept_df.crossJoin(total_df).withColumn(
    "percentage",
    (F.col("total_sal")/F.col("overall_sal"))*100
)
result_df.orderBy(F.col("percentage").desc()).show()

+----------+---------+-----------+------------------+
|department|total_Sal|overall_sal|        percentage|
+----------+---------+-----------+------------------+
|        IT|   240000|     562000|42.704626334519574|
|        HR|   175000|     562000| 31.13879003558719|
|     Sales|   147000|     562000|26.156583629893237|
+----------+---------+-----------+------------------+



In [ ]:
# City wise salary Ranking 
# Sal distribution by city - 
    #for each city - min sal, max_sal and avg sal 
# compare department vs company avegare 

    # Find avg per dept, compare with global org, return department above avg